#### Build a simple LLM application with LCEL

In the quickstart, we'll show you how to build a simple LLM application with Langchain. This application will translate text from English into anpther Language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with Langchain - a lot of features can be built with just some prompting and an LLM call!!

After seeing this video, you'll have a high level overview of:
* Using Language Models
* Using Prompt Templates and OutputParsers
* Using Langchain Expression Language (LCEL) to chain components together.
* Debugging and tracing your application using Langsmith
* Deploying your application with LangServe

In [27]:
# Open source models usig Groq API

import os

from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key = os.getenv("GROQ_API_KEY")

In [28]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001A9063D9E80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A90608A600>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [29]:
from langchain_core.messages import HumanMessage, SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French. Don't return anything extra"),
    HumanMessage(content="Hello how are you?")
]

result = model.invoke(messages)

In [30]:
## For finding out which models in groq do I have access for

# from groq import Groq
# import os

# client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# models = client.models.list()

# for model in models.data:
#     print(f"Model: {model.id}")
#     print(f"Owner: {model.owned_by}")
#     print(f"Context Window: {getattr(model, 'context_window', 'N/A')}")
#     print("-" * 50)

In [31]:
result

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "Hello how are you?"\n   - Task: Translate from English to French\n   - Constraint: "Don\'t return anything extra"\n\n2.  **Identify Key Components:**\n   - "Hello" -> Bonjour / Salut\n   - "how are you?" -> comment allez-vous ? / comment ça va ? / comment vas-tu ?\n   - Context: General greeting, neutral/formal is usually safest unless specified otherwise. "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?" are both acceptable. I\'ll go with a standard, widely used translation: "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?" Given the informal "Hello how are you?", "Bonjour, comment ça va ?" is very natural. However, "Bonjour, comment allez-vous ?" is more formal. I\'ll stick with a standard, polite version: "Bonjour, comment allez-vous ?" or simply "Bonjour, comment ça va ?". Actually, the most direct and common translation is "Bonjour, comment allez-vous

In [32]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "Hello how are you?"\n   - Task: Translate from English to French\n   - Constraint: "Don\'t return anything extra"\n\n2.  **Identify Key Components:**\n   - "Hello" -> Bonjour / Salut\n   - "how are you?" -> comment allez-vous ? / comment ça va ? / comment vas-tu ?\n   - Context: General greeting, neutral/formal is usually safest unless specified otherwise. "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?" are both acceptable. I\'ll go with a standard, widely used translation: "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?" Given the informal "Hello how are you?", "Bonjour, comment ça va ?" is very natural. However, "Bonjour, comment allez-vous ?" is more formal. I\'ll stick with a standard, polite version: "Bonjour, comment allez-vous ?" or simply "Bonjour, comment ça va ?". Actually, the most direct and common translation is "Bonjour, comment allez-vous ?" or "Salut, com

In [33]:
### Using LCEL - chain the components
chain = model|parser
chain.invoke(messages)

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Source text: "Hello how are you?"\n   - Target language: French\n   - Constraint: "Don\'t return anything extra"\n\n2.  **Translate to French:**\n   - "Hello" -> "Bonjour"\n   - "how are you?" -> "comment allez-vous ?" (formal/plural) or "comment vas-tu ?" (informal/singular) or "comment ça va ?" (common/informal)\n   - Standard/neutral translation: "Bonjour, comment allez-vous ?" or "Bonjour, comment ça va ?"\n   - I\'ll go with a standard, polite, and commonly used version: "Bonjour, comment allez-vous ?" or simply "Bonjour, comment ça va ?"\n   - Given the context is just a simple greeting, "Bonjour, comment allez-vous ?" is safe and accurate. Alternatively, "Bonjour, comment ça va ?" is also very common. I\'ll stick with "Bonjour, comment allez-vous ?" as it\'s a direct translation of the standard phrase.\n\n   Actually, a more direct and natural translation for everyday use is "Bonjour, comment ça va ?" o

In [34]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}:"
prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template),
    ("user", "{text}")]
)

In [35]:
result = prompt.invoke({"language": "French", "text":"hello"})

In [36]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hello', additional_kwargs={}, response_metadata={})]

In [37]:
chain = prompt | model | parser
chain.invoke({"language":"French", "text":"hello"})

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User wants to translate "hello" into French.\n   - Input: "hello"\n   - Target language: French\n\n2.  **Identify Key Translation:**\n   - "hello" in English translates to several options in French depending on context:\n     - "bonjour" (standard, daytime, formal/informal)\n     - "salut" (informal, like "hi")\n     - "coucou" (very informal, affectionate)\n   - The most common and appropriate default translation is "bonjour".\n\n3.  **Formulate Response:**\n   - Provide the direct translation.\n   - Optionally note context/usage if helpful, but keep it concise as requested.\n   - Response: "Bonjour" (or "Salut" for informal)\n\n   I\'ll stick to the standard/most common: "Bonjour". I can also add a brief note about informal usage if needed, but the prompt is simple.\n\n   Draft: "Bonjour" (or "Salut" in informal contexts).\n\n4.  **Final Output Generation:**\n   - Keep it direct and accurate.\n   - "Bonjour"